In [ ]:
import os
import sys
import json
import pickle
import importlib
import itertools
import numpy as np
import pandas as pd
import nibabel as nib

sys.path.append('/host/verges/tank/data/daniel/00_commonUtils/00_code/genUtils/')
import t1

import visUtils
import projectUtils as prjUtils

# TODO. AP coordinates, subtract all values by min

In [ ]:
# parameters

demo_dicts_json_pth = "/host/verges/tank/data/daniel/04_inVivoHistology/outputs/grp_dicts_26Feb2026-1503.json"

dirs_project = {
    'dir_root': '/host/verges/tank/data/daniel/04_inVivoHistology',
    'dir_data': 'data/',
    'dir_out': 'outputs/',
    'demo_dfs': 'demo_dfs'
}

analysis_params = {
    
    'time': '26Feb2026-1503', # for file naming
    
    'verbose': True,
    'qMap_names':["T1map"], # names of qMAP of interest
    
    'ctrl_grp':"CTRL_match", # reference to Z score on
    'test_grps': ["TLE_L", "TLE_R", "TLE_LR"], # groups to compare to reference for Z score (excluding the ctrl_grp)
    
    'icFlip': True,
    'ipsiTo': 'L', 
    
    'mapDate': '12Feb2026-1304', # date of map creation, for finding the appropriate maps
    'nSurfs':16,
    'smoothing':["0", "0p5"],

    'equiVol_str': "equivol",
}

visualization_params = {}


study_dicts = [
    { # MICS
        'studyName': 'MICs',
        'studyDescrip': '3T',
        'dir_root': '/data/mica3/BIDS_MICs/',
        'dir_raw': 'rawdata/',
        'dir_deriv': 'derivatives/',
        'dir_fs': 'freesurfer/',
        'dir_mp': 'micapipe_v0.2.0/',
        'dir_hu': 'hippunfold_v1.3.0/hippunfold/', # update to v2?
    },
    { # PNI
        'studyName': 'PNI',
        'studyDescrip': '7T',
        'dir_root': '/data/mica3/BIDS_PNI/',
        'dir_raw': 'rawdata/',
        'dir_deriv': 'derivatives/',
        'dir_fs': 'fastsurfer/',
        'dir_mp': 'micapipe_v0.2.0/',
        'dir_hu': 'hippunfold_v1.3.0/hippunfold/', # update to v2?

        'dir_root_pilot': '/host/verges/tank/data/MICA-7T-pilot/',
        'dir_deriv_pilot': 'derivatives/',
        'dir_mp_pilot': 'micapipe/',
    }
]

In [14]:
# Demographic summary
print(f"\nDemographic summary ({demo_dicts_json_pth}):")
with open(demo_dicts_json_pth, 'r') as f:
    dict_list = json.load(f)

list_grps = [[d['group'], pd.read_csv(d['demo_df_pth'])] for d in dict_list] # list of [names, demo_df]
t1.summarize_grps(list_grps, print=False)

,Group,n_unique,n_SES,Sex_percF,Sex_F,Sex_M,Age_min,Age_25p,Age_mean,Age_median,Age_75p,Age_max,Age_std,Age_IQR
0,CTRL,46.0,46.0,52.173913,24.0,22.0,20.2,26.50,29.530000,27.95,31.475,42.8,5.362228,4.975
1,TLE_L,10.0,10.0,50.000000,5.0,5.0,29.8,30.75,36.510000,32.65,36.500,61.2,9.687730,5.750
2,TLE_R,10.0,10.0,90.000000,9.0,1.0,20.4,24.10,34.120000,33.65,42.925,49.7,10.824335,18.825
3,TLE_LR,20.0,20.0,70.000000,14.0,6.0,20.4,30.40,35.315000,32.65,42.175,61.2,10.072698,11.775
4,CTRL_match,30.0,30.0,53.333333,16.0,14.0,24.5,26.95,31.116667,29.70,34.550,42.8,5.546735,7.600


In [ ]:
# Overlay surfaces on coronal section of volume
sub = "PNC039"
ses = "a1"
hemi = "L"
surfDate = "13Mar2026"
volName = "T1w"
#visUtils.showSurfsOnVol(sub = sub, ses=ses, volName=volName,dirs_project=dirs_project,analysis_params=analysis_params, study_dict=study_dicts[1])
def getSurf_names(dir_pth:str, file_ptrn:str):
    surf_files = [os.path.join(dir_pth, f) for f in os.listdir(dir_pth) if file_ptrn in f]
    surf_files.sort()
    return surf_files
root = f"/host/verges/tank/data/daniel/04_inVivoHistology/data/PNI/sub-{sub}_ses-{ses}/surfs" 
getSurf_names(root, f"sub-{sub}_ses-{ses}_hemi-{hemi}_{surfDate}_equivol-")
# vol:
# surfs:

In [ ]:
# visualize raw values and stat-z by individual  -- collapsing along axes
importlib.reload(prjUtils)
importlib.reload(visUtils)

# TODO. IMPLEMENT LABELLING and CORRECT Y-LIMITS

analysis_params['icFlip'] = True
#grps_to_vis = analysis_params['test_grps'] + [analysis_params['ctrl_grp']]
grps_to_vis = ['TLE_LR']

#surf_range = range(1, analysis_params['nSurfs'] + 1)
# surf_range = range(1, analysis_params['nSurfs'] + 1, 4) # with a step
surf_range = range(3, 2) # with a step
surf_range = [8]
#data_stats = ['raw', 'z']
data_stats = ['z']

#smoothing_to_plot = analysis_params['smoothing']
smoothing_to_plot = ["0p5"]

axis = 'AN'
collapse_stat = 'mean'
bin_width_mm = 0.1

parc_label_pth = "/host/verges/tank/data/daniel/04_inVivoHistology/code/resources/13Feb/stitch_lblVals_ctx-glsr_hipp-DK25_12Feb2026_mask-mTemp.label.gii"
parc_label_csv_pth = "/host/verges/tank/data/daniel/04_inVivoHistology/code/resources/13Feb/stitch_lblValDetails_ctx-glsr_hipp-DK25_12Feb2026-0915.csv"

for grp, mapName, stat, lvl, smth in itertools.product(grps_to_vis, analysis_params['qMap_names'], data_stats, surf_range, smoothing_to_plot):  # get aggregate map path
    print(f"{grp} | {stat} surf:{lvl}/{analysis_params['nSurfs']} smth-{smth}mm")

    dir = prjUtils.get_aggregateMapDir(dirs_project, analysis_time=analysis_params['time'], grp=grp)
    if grp == analysis_params['ctrl_grp']:
        hemis = ['L', 'R']
    elif analysis_params['icFlip']:
        hemis = ['ipsi', 'contra']
    else:
        hemis = ['L', 'R']
    
    for hemi in hemis:
        if stat in ['raw', '']:
            ylim = (1000, 2800)
            _, file = prjUtils.get_aggregateMapName(dirs_project=dirs_project, 
                                                       grp=grp, 
                                                       hemi=hemi, 
                                                       equiVol_str=analysis_params['equiVol_str'], 
                                                       lvl=lvl, nSurfs=analysis_params['nSurfs'], 
                                                       mapName=mapName, smth=smth, analysis_time=analysis_params['time'])
        elif stat == 'z':
            if collapse_stat in ['mean', 'median']:
                ylim = (-4,4)
            elif collapse_stat in ['std', 'skew', 'kurt']:
                ylim = (0, 4)
            elif collapse_stat in ['min', 'max', '90perc', '90th percentile', '10perc', '10th percentile']:
                ylim = (-10, 10)
            else:
                ylim = (-2,2)
            file = prjUtils.get_statSummaryName(grp=grp, hemi=hemi, 
                                                    map_date=analysis_params['time'], 
                                                    equiVol_str=analysis_params['equiVol_str'], 
                                                    lvl=lvl, nSurfs=analysis_params['nSurfs'], 
                                                    mapName=mapName, smth=smth, stat_name=stat, ext=".parquet")

        file_pth = os.path.join(dir, file)
        data = pd.read_parquet(file_pth)
        print(f"\t{hemi}: {data.shape}")
                
        visUtils.plot_collapsed_stats(data, 
                                      axis=axis, 
                                      stat=collapse_stat, 
                                      bin_width_mm=bin_width_mm, 
                                      anat_label_gii_pth=parc_label_pth,
                                      anat_label_csv_pth=parc_label_csv_pth,
                                      label_method='mode',
                                      title = f"{grp}-{hemi} | ({stat}) surf:{lvl}/{analysis_params['nSurfs']} smth-{smth}mm", 
        )
        break

In [132]:
ap = nib.load("/host/verges/tank/data/daniel/04_inVivoHistology/code/resources/13Feb/stitch_lbl_AP_masked-13Feb2026.label.gii").darrays[0].data

In [133]:
ap

array([66.815155, 65.229614, 63.758728, ..., 81.420074, 80.89905 ,
       80.60808 ], shape=(8477,), dtype=float32)